<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "776"
iso_country = "Tonga"
year = 2025
import_portfolio = "h5_ag6_by_sea_FM.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 679.220*1e6

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Tonga': 

Number of trade records present in dataset: 3649
Number of partner countries present in dataset: 42
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$65.8m
Total weight within the dataset: 81.5 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,776,Tonga,764,Thailand,Foreign Import,Sea,852380,852380 | Media n.e.c. in head,8523,"8523 | Discs, tapes, solid-",85,85 | Electrical machinery,4752,"Records, tapes and other recorded media for so...",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,1647.200,NaN,1647.200,u,87.467,31.529,16.472000,10009.0,6219.302339,102.444348,['Container'],['Container'],0.535459,0.534524,0.534616,0.534616,0.535035,0.535035,0.560117,0.992704,0.617828,0.573595,0.575153,0.589558,0.548453,0.972201,0.650302,0.595667,0.581998,0.608357,0.538223,1.057203,0.806406,0.705254,0.652494,0.634088,0.528183,1.124066,0.924977,0.866782,0.698579,0.679583,0.518332,1.139083,1.088293,1.097651,0.757332,0.713880,0.508284,1.115671,1.069154,1.171416,0.866488,0.726965,0.498817,0.995205,1.037269,1.086950,0.814812,0.735554,0.489165,0.914181,0.929289,1.159168,0.957636,0.740425,4752
1,1,776,Tonga,764,Thailand,Foreign Import,Sea,852872,852872 | Reception apparatus,8528,8528 | Monitors and project,85,85 | Electrical machinery,47313,"Television receivers, whether or not combined ...",Electrical and Machinery,40,ELE - Electronic equipment,1242.955,NaN,1242.955,u,3.971,47.013,36.915763,10009.0,6219.302339,229.590294,['Container'],['Container'],1.200029,1.197934,1.198139,1.198139,1.199078,1.199078,1.255292,2.224771,1.384627,1.285497,1.288988,1.321272,1.229151,2.178822,1.457407,1.334963,1.304328,1.363401,1.206224,2.369321,1.807255,1.580561,1.462318,1.421068,1.183722,2.519169,2.072987,1.942565,1.565600,1.523029,1.161645,2.552825,2.438999,2.459969,1.697273,1.599893,1.139126,2.500355,2.396106,2.625286,1.941906,1.629217,1.117910,2.230377,2.324646,2.435987,1.826093,1.648467,1.096278,2.048792,2.082650,2.597838,2.146178,1.659383,47313


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $26.8m (40.7%), ... 

                            cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                          
04 | Dairy produce; birds  8603352.4   5916340.7     13.1      7.3
22 | Beverages, spirits a  3315375.7   4576716.1      5.0      5.6
19 | Preparations of cere  2664937.4   1086640.1      4.0      1.3
21 | Miscellaneous edible  2529385.9    783555.8      3.8      1.0
23 | Food industries, res  2152136.3   1592365.6      3.3      2.0
11 | Products of the mill  1747234.6   4245669.9      2.7      5.2
02 | Meat and edible meat  1522316.1    560003.5      2.3      0.7
20 | Preparations of vege   897982.7    639791.4      1.4      0.8
16 | Meat, fish, crustace   871677.5    193987.0      1.3      0.2
17 | Sugars and sugar con   743119.2   1273639.6      1.1      1.6
08 | Fruit and nuts, edib   525042.7    441413.0      0.8      0.5
15 | Animal, vegetable or   4

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $25.1m (38.1%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Food & Beverages  12585314.5   9032096.7     19.1     11.1
Agriculture       12499970.4  11251921.9     19.0     13.8
Fishing              17895.8      5358.0      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $21.9m (33.3%), ... 

                                cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                             
MIL - Dairy products           9305206.6   5663643.4     14.1      6.9
OFD - Food products n.e.c.     9104617.9   8059266.8     13.8      9.9
OMT - Meat products n.e.c.     1532855.2    596951.5      2.3      0.7
CMT - Bovine meat prods         679379.6    123614.5      1.0      0.2
V_F - Vegetables, fruit, nuts   550927.0    609883.2      0.8      0.7
VOL - Vegetable oils and fats   473435.9    316606.1      0.7      0.4
SGR - Sugar                     115902.4    251670.8      0.2      0.3
PCR - Processed rice             87194.9    151457.5      0.1      0.2
OCR - Crops n.e.c.               74212.0     24932.2      0.1      0.0
GRO - Cereal grains n.e.c.       15803.0     65369.0      0.0      0.1
OAP - Animal products n.e.c.      3738.6      1065.9      0.0      0.0
PFB - Plant-

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 40.7% 
GTAP 33.3% 
EORA 38.1%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27859/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,776,Tonga,764,Thailand,Foreign Import,Sea,852380,852380 | Media n.e.c. in head,8523,"8523 | Discs, tapes, solid-",85,85 | Electrical machinery,4752,"Records, tapes and other recorded media for so...",Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,1647.200,NaN,1647.200,u,87.467,31.529,16.472000,10009.0,6219.302339,102.444348,['Container'],['Container'],0.535459,0.534524,0.534616,0.534616,0.535035,0.535035,0.560117,0.992704,0.617828,0.573595,0.575153,0.589558,0.548453,0.972201,0.650302,0.595667,0.581998,0.608357,0.538223,1.057203,0.806406,0.705254,0.652494,0.634088,0.528183,1.124066,0.924977,0.866782,0.698579,0.679583,0.518332,1.139083,1.088293,1.097651,0.757332,0.713880,0.508284,1.115671,1.069154,1.171416,0.866488,0.726965,0.498817,0.995205,1.037269,1.086950,0.814812,0.735554,0.489165,0.914181,0.929289,1.159168,0.957636,0.740425,4752,NaN
1,1,776,Tonga,764,Thailand,Foreign Import,Sea,852872,852872 | Reception apparatus,8528,8528 | Monitors and project,85,85 | Electrical machinery,47313,"Television receivers, whether or not combined ...",Electrical and Machinery,40,ELE - Electronic equipment,1242.955,NaN,1242.955,u,3.971,47.013,36.915763,10009.0,6219.302339,229.590294,['Container'],['Container'],1.200029,1.197934,1.198139,1.198139,1.199078,1.199078,1.255292,2.224771,1.384627,1.285497,1.288988,1.321272,1.229151,2.178822,1.457407,1.334963,1.304328,1.363401,1.206224,2.369321,1.807255,1.580561,1.462318,1.421068,1.183722,2.519169,2.072987,1.942565,1.565600,1.523029,1.161645,2.552825,2.438999,2.459969,1.697273,1.599893,1.139126,2.500355,2.396106,2.625286,1.941906,1.629217,1.117910,2.230377,2.324646,2.435987,1.826093,1.648467,1.096278,2.048792,2.082650,2.597838,2.146178,1.659383,47313,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $17.5m of $65.8m (26.6%)
Volume: 14.8 of 81.5 million kilograms (18.2%)
Value-to-Weight: 1.18 from 0.81 US$ per kg (+46.6%)
Average distance to market (volume-weighted): 1972.6 from 2862.0 nm (+-31.1%)

            food_group  N_com   cifvalue  vol_kg_vtw  pct_val  pct_vol
0                Dairy     28  8425350.1   5817305.1     12.8      7.1
1    Processed Cereals    138  5573386.5   5755956.6      8.5      7.1
2        Meat and Prep     20  1128020.4    478087.4      1.7      0.6
3      Sugar and Honey     36   771396.6   1281746.7      1.2      1.6
4           Vegetables     43   672833.4    709735.5      1.0      0.9
5               Fruits     29   503900.1    431846.0      0.8      0.5
6        Fats and Oils     16   437811.0    274942.7      0.7      0.3
7  Unprocessed Cereals      5    15803.0     65369.0      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,4435221.3,7843891.5,4888571.5,4540646.0,4552527.7,4666965.9,4105995.4,8998829.9,8599845.9,8675780.0,5989043.5,5646142.5,3876187.2,7228602.6,7346806.5,9151752.3,7562431.4,5855158.1
1,Fats and Oils,254858.3,449741.1,280687.8,260814.1,261479.1,268090.7,236035.3,515810.7,493069.0,497600.2,343683.3,324047.8,222896.3,414718.9,421478.7,524286.0,433317.0,335989.3
2,Fruits,458385.4,798935.7,502696.9,468369.1,469278.8,481367.7,425516.5,915384.4,876390.2,885530.5,613498.8,578876.5,402573.2,739898.9,751113.5,926892.2,767193.4,599652.9
3,Meat and Prep,1257240.4,2228224.5,1386776.5,1287492.9,1290989.5,1323323.4,1163448.8,2556788.4,2442785.0,2463788.3,1699908.0,1602377.1,1097979.8,2051972.5,2085883.7,2601870.8,2149510.3,1661959.4
4,Processed Cereals,5922923.1,10338108.6,6497232.1,6050646.0,6064053.5,6220841.8,5496467.9,11864011.5,11351086.3,11472760.7,7944747.0,7485057.2,5198791.3,9576331.9,9728551.2,12008867.8,9941359.7,7753920.2
5,Sugar and Honey,1357616.0,2185595.8,1442730.8,1361541.7,1365309.8,1409505.4,1276713.2,2569393.3,2458623.1,2530415.8,1783536.7,1639665.3,1220241.0,2113326.9,2159979.0,2493107.5,2098716.3,1685450.1
6,Unprocessed Cereals,159006.7,277629.0,174330.6,162253.5,162713.3,166991.5,147526.9,319981.5,305708.8,309369.9,214155.1,200929.4,139513.7,257672.4,262237.0,323182.9,267780.3,208104.2
7,Vegetables,895973.9,1554236.7,978592.1,912984.0,915125.8,939530.3,832733.9,1791319.1,1710299.0,1733326.2,1195091.1,1127917.1,787675.5,1441794.6,1466665.1,1792379.0,1482115.5,1167796.9


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.40,0.05,0.01,0.01,0.03,0.58,0.53,0.54,0.22,0.18,0.40,0.41,0.63,0.44,0.23
1,Fats and Oils,0.45,0.06,0.01,0.02,0.03,0.64,0.59,0.60,0.25,0.20,0.44,0.45,0.69,0.48,0.26
2,Fruits,0.68,0.09,0.02,0.02,0.05,0.97,0.89,0.91,0.37,0.30,0.67,0.69,1.04,0.72,0.39
3,Meat and Prep,0.86,0.11,0.03,0.03,0.06,1.24,1.13,1.15,0.48,0.39,0.85,0.88,1.33,0.93,0.50
4,Processed Cereals,0.79,0.10,0.02,0.03,0.05,1.14,1.05,1.07,0.44,0.36,0.79,0.81,1.22,0.85,0.46
5,Sugar and Honey,1.07,0.11,0.01,0.01,0.07,1.68,1.53,1.63,0.66,0.47,1.16,1.22,1.65,1.14,0.60
6,Unprocessed Cereals,7.51,0.97,0.21,0.23,0.51,10.91,10.01,10.24,4.22,3.38,7.48,7.77,11.62,8.12,4.34
7,Vegetables,0.98,0.12,0.03,0.03,0.06,1.42,1.30,1.34,0.54,0.44,0.97,1.01,1.49,1.03,0.56
